# 04 — Klemeš differential split-sample validation

Klemeš (1986): a model calibrated on one climatic regime should be validated on a
*different* one. Here GR4J is calibrated on the wettest contiguous block of years and
validated on the driest, and vice-versa.

The one piece this needs beyond notebook 02 is **warm-up handling per split**: a
validation window that does not start at the record beginning has cold production /
routing stores, so `klemes_warmup` runs the model over a warm-up slice *immediately
before* each window and scores only the window itself.

PET = Morton (same as 02/03).

In [1]:
import import_ipynb
from GR4J import compute_Q
from Metric_Calculation import compute_KGE, compute_NSE, compute_PBIAS
from scipy.optimize import differential_evolution
import numpy as np
import pandas as pd

2.28 ms ± 63.8 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
5.684341886080802e-14
[0.67712195 0.62997907 0.58832275 ... 0.60016149 0.58716065 0.92408118]


In [2]:
csv_path = r"D:\Claude\flood_hydrology_modeling\data\processed\prec_PET_sf.csv"
df = pd.read_csv(csv_path)
P     = df['prec_AGCD'].to_numpy(dtype=np.float64)
E     = df['PET_morton'].to_numpy(dtype=np.float64)   # Morton PET (placeholder)
obs   = df['sf_mmd'].to_numpy(dtype=np.float64)        # keep NaN as-is
dates = pd.to_datetime(df['date'])
years = dates.dt.year.to_numpy()
bounds = [(1, 1500), (-8, 8), (1, 500), (0.5, 4)]      # x1, x2, x3, x4  (x2 widened -8..8 to test bound-pinning)
WARMUP = 730                                           # 2-year warm-up per split
print(f"record: {dates.min().date()} -> {dates.max().date()}  ({len(df)} days)")

record: 1970-05-28 -> 2021-06-30  (18662 days)


In [3]:
def klemes_warmup(params, eval_start, eval_end, warmup_days=WARMUP):
    """Score GR4J on obs[eval_start:eval_end] after a warm-up run of `warmup_days`
    immediately preceding the window, so the stores are not cold at window start.

    Returns [KGE, r, alpha, beta] over the finite observed days in the window.
    """
    sim_start = max(0, eval_start - warmup_days)
    sim = compute_Q(P[sim_start:eval_end], E[sim_start:eval_end], params)
    off = eval_start - sim_start          # where the eval window begins inside `sim`
    s   = sim[off:]
    o   = obs[eval_start:eval_end]
    mask = ~np.isnan(o)
    return compute_KGE(o[mask], s[mask])

In [4]:
# --- pick the wettest and driest contiguous multi-year blocks -------------
ann_P = pd.Series(P, index=years).groupby(level=0).sum()   # annual precip (mm/yr)
K = max(5, len(ann_P) // 3)                                # block length in years (adjustable)
roll = ann_P.rolling(K).sum()                              # K-year running total, labelled by END year
wet_end, dry_end = int(roll.idxmax()), int(roll.idxmin())
wet_years = (wet_end - K + 1, wet_end)
dry_years = (dry_end - K + 1, dry_end)

def year_span(y0, y1):
    idx = np.where((years >= y0) & (years <= y1))[0]
    return int(idx[0]), int(idx[-1]) + 1                   # [start, end) row indices

wet = year_span(*wet_years)
dry = year_span(*dry_years)
print(f"K = {K} years")
print(f"WET block  {wet_years[0]}-{wet_years[1]}  P={ann_P.loc[wet_years[0]:wet_years[1]].mean():.0f} mm/yr")
print(f"DRY block  {dry_years[0]}-{dry_years[1]}  P={ann_P.loc[dry_years[0]:dry_years[1]].mean():.0f} mm/yr")

K = 17 years
WET block  1973-1989  P=2128 mm/yr
DRY block  2001-2017  P=1810 mm/yr


In [5]:
def calibrate(eval_start, eval_end, seed=42):
    """DE calibration on one window, warm-up aware, KGE objective."""
    def objective(x):
        k = klemes_warmup(x, eval_start, eval_end)[0]
        return 1e6 if not np.isfinite(k) else -k
    res = differential_evolution(objective, bounds, seed=seed, maxiter=300,
                                 popsize=15, tol=1e-6, polish=True, workers=1)
    return res.x, -res.fun

# calibrate on each regime ...
theta_wet, kge_wet = calibrate(*wet)
theta_dry, kge_dry = calibrate(*dry)

# ... and cross-validate on the other regime (Klemeš differential split-sample)
# validation on the OTHER regime returns full KGE decomposition [KGE, r, alpha, beta]:
rows = []
for name, th, kc, val_win in [("wet -> dry", theta_wet, kge_wet, dry),
                              ("dry -> wet", theta_dry, kge_dry, wet)]:
    vkge, vr, va, vb = klemes_warmup(th, *val_win)
    rows.append((name, th, kc, vkge, vr, va, vb))

# val_r = correlation, val_a = variability ratio (sd_sim/sd_obs), val_b = bias ratio (mean_sim/mean_obs)
print(f"{'scheme':11s} {'x1':>7} {'x2':>6} {'x3':>7} {'x4':>5}  "
      f"{'calKGE':>7} {'valKGE':>7} {'val_r':>6} {'val_a':>6} {'val_b':>6}")
for name, th, kc, vkge, vr, va, vb in rows:
    print(f"{name:11s} {th[0]:7.1f} {th[1]:+6.2f} {th[2]:7.1f} {th[3]:5.2f}  "
          f"{kc:7.3f} {vkge:7.3f} {vr:6.3f} {va:6.3f} {vb:6.3f}")

scheme           x1     x2      x3    x4   calKGE  valKGE  val_r  val_a  val_b
wet -> dry     56.6  -8.00   423.7  1.00    0.830   0.478  0.910  0.630  0.642
dry -> wet    138.7  +4.15   121.5  1.06    0.907   0.229  0.823  1.519  1.543
